In [ ]:
import pandas as pd
from kaggle.api.kaggle_api_extended import KaggleApi

In [ ]:
import importlib
import os
import sys

sys.path.insert(0, os.path.abspath(".."))
from src.data_cleaning.data import _get_project_data_dir, download_data, load_data, clean_data

In [ ]:
download_data()

In [ ]:
df = load_data()
df_cleaned = clean_data(df)

In [ ]:
# pip install nltk si besoin
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [ ]:
_stop_words = set(stopwords.words("english"))
_lemmatizer = WordNetLemmatizer()


def preprocess_text(text, remove_stopwords=True, remove_punctuation=True, lemmatize=True):
    tokens = word_tokenize(text)
    tokens = [t.lower() for t in tokens]

    if remove_punctuation:
        tokens = [t for t in tokens if t.isalpha()]

    if remove_stopwords:
        tokens = [t for t in tokens if t not in _stop_words]

    if lemmatize:
        for pos in ("n", "v", "a"):
            tokens = [_lemmatizer.lemmatize(t, pos=pos) for t in tokens]

    return " ".join(tokens)

In [ ]:
_stop_words
_stop_words.remove()

In [ ]:
text = "I ate chocolate yesterday and I feel happy right now!"
preprocess_text(text)

In [ ]:
# On sauvegarde en .csv
# df_balanced.to_csv("../../data/emotion_balanced.csv", index=False)

In [ ]:
# On sauvegarde la data en .pkl pour éviter de refaire le nettoyage à chaque fois
# df_balanced.to_pickle("../data/emotion_balanced.pkl")

In [ ]:
######## ON REFAIT TOUT SANS BALANCE CLASSES
######## ET AVEC CLASS_WEIGHT A LA FIN

In [ ]:
df2 = load_data()
df2.shape

In [ ]:
df2_cleaned = clean_data(df2)

In [ ]:
df2_cleaned.shape

In [ ]:
sys.path.insert(0, os.path.abspath(".."))
from src.training.preprocess import preprocess_text

In [ ]:
# On applique clean et preprocess_text sur title
df2_cleaned["clean_title"] = df2_cleaned["title"].apply(preprocess_text)
df2_cleaned["clean_title"] = df2_cleaned["clean_title"].apply(preprocess_text)

In [ ]:
df2_cleaned.drop(columns=["title"], inplace=True)

In [ ]:
df2_cleaned.shape

In [ ]:
# df2_cleaned.to_csv("../../data/emotion_preprocessed.csv", index=False)

In [ ]:
df2_cleaned = pd.read_csv("../../data/emotion_preprocessed.csv")

In [ ]:
# Je ne sais pas pourquoi il reste des NA
df2_cleaned.dropna(inplace=True)

In [ ]:
df2_cleaned.isna().sum()

In [ ]:
"""# TODO: compute the BOW
from sklearn.feature_extraction.text import CountVectorizer
from scipy import sparse

# We create the output BOW, we can even reject directly the stop words and the punctuation, how magical?
vectorizer = CountVectorizer(max_features=10000,
                             #stop_words='english',
                             #min_df=0.001, max_df=0.999
)
X = vectorizer.fit_transform(df2_cleaned["clean_title"])  # scipy sparse matrix
print(type(X), X.shape)  # should be (num_docs, num_features)

BOW = pd.DataFrame.sparse.from_spmatrix(
    X,
    index=df2_cleaned.index,
    columns=vectorizer.get_feature_names_out(),
)"""

In [ ]:
from sklearn.model_selection import train_test_split

X = df2_cleaned["clean_title"]
y = df2_cleaned["label"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Fit TF-IDF on training text only, then transform test text.
vectorizer = TfidfVectorizer(
    max_features=10000,
    # stop_words='english',
    # min_df=0.001, max_df=0.999
)

X_train = vectorizer.fit_transform(X_train)
X_test = vectorizer.transform(X_test)
print(type(X_train), X_train.shape, X_test.shape)

BOW_train = pd.DataFrame.sparse.from_spmatrix(
    X_train,
    index=y_train.index,
    columns=vectorizer.get_feature_names_out(),
)

In [ ]:
# Final training/evaluation matrix shapes
X_train.shape, X_test.shape, y_train.shape, y_test.shape

In [ ]:
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(max_iter=1000, class_weight="balanced")
log_reg.fit(X_train, y_train)
log_reg.score(X_test, y_test)

In [ ]:
from sklearn.metrics import classification_report

class_report = classification_report(y_test, log_reg.predict(X_test))

In [ ]:
# Classification report en tableau
print(class_report)

In [ ]:
import joblib

# joblib.dump(log_reg, '../models/depression_log_classifier.pkl')
# joblib.dump(vectorizer, '../models/tfidf_vectorizer.pkl')
print("✅ Modèle baseline sauvegardé !")

In [ ]:
import joblib

vectorizer = joblib.load("../models/tfidf_vectorizer.pkl")
log_reg = joblib.load("../models/depression_log_classifier.pkl")

In [ ]:
from sklearn.metrics import confusion_matrix, roc_curve, auc
import seaborn as sns

import matplotlib.pyplot as plt

# Predictions for confusion matrix and ROC
y_pred = log_reg.predict(X_test)
y_score = log_reg.predict_proba(X_test)[:, 1]

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_score)
roc_auc = auc(fpr, tpr)

# Plot side by side
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion matrix heatmap
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    cbar=False,
    xticklabels=["Pred 0", "Pred 1"],
    yticklabels=["True 0", "True 1"],
    ax=axes[0],
)
axes[0].set_title("Confusion Matrix")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("Actual")

# ROC curve
axes[1].plot(fpr, tpr, label=f"AUC = {roc_auc:.4f}")
axes[1].plot([0, 1], [0, 1], "k--")
axes[1].set_title("ROC Curve")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].legend(loc="lower right")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np

X_sample = X_train[np.random.choice(X_train.shape[0], 20000, replace=False)]

In [ ]:
import shap

explainer = shap.Explainer(log_reg, X_sample)
shap_values = explainer(X_sample)

In [ ]:
# Use the vectorizer vocabulary so SHAP labels display tokens instead of generic feature indices.
feature_names = vectorizer.get_feature_names_out().tolist()
shap_values.feature_names = feature_names
shap.plots.bar(shap_values[0], max_display=20)

In [ ]:
# summary_plot needs SHAP values, not the explainer object.
shap.summary_plot(shap_values, X_sample, feature_names=feature_names, plot_size=(20, 10), max_display=20)

In [ ]:
ind = 10
shap.plots.force(shap_values[ind], matplotlib=True)

In [ ]:
ind = 20
shap.plots.waterfall(shap_values[ind, :], max_display=20)

In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath(".."))
from src.dashboard.shap import shap_graph

shap_graph(log_reg, vectorizer, X_sample)

In [ ]:
import re

words = re.findall(r"\w+|\W+", "I want to eat chocolate")
words